# Set-Up

## Data Loading and Library Installation

In [1]:
from IPython.display import clear_output

# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
!pip install imutils numpy easyocr iptcinfo3 opencv-python face_recognition
!pip install git+https://github.com/james-see/iptcinfo3.git@master
!pip install Pillow==9.5.0
clear_output()

  Using cached opencv_python-4.9.0.80-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tifffile-2024.2.12-py3-none-any.whl.metadata (31 kB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\users\\xapha\\anaconda3\\lib\\site-packages\\typing_extensions-3.10.0.2.dist-info\\METADATA'



  Cloning https://github.com/james-see/iptcinfo3.git (to revision master) to c:\users\xapha\appdata\local\temp\pip-req-build-fs_kl4np
  Resolved https://github.com/james-see/iptcinfo3.git to commit dfbfd902f64205ad42517dce288b46e06bc4b585
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/james-see/iptcinfo3.git 'C:\Users\Xapha\AppData\Local\Temp\pip-req-build-fs_kl4np'


In [ ]:
!touch ./__init__.py
!mkdir ./textrecog/
!touch /content/textrecog/__init__.py

'touch' is not recognized as an internal or external command,
operable program or batch file.
The syntax of the command is incorrect.
'touch' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import re
import os
from tqdm import tqdm
import json
import pandas as pd
import random
import shutil
from pathlib import Path

import easyocr
import sklearn

from iptcinfo3 import IPTCInfo

import face_recognition # Needs to be run with CUDA
import cv2
import numpy as np

# # From: kapil-varshney/text-detection-recognition.git
# from imutils.object_detection import non_max_suppression
# import numpy as np
# import pytesseract
# import imutils
# import argparse
# import cv2

### Goofing around with Facial Recognition

In [ ]:
# Define arguments
args = {}
image_path = "./images"

# args['width'] = 800 # nearest multiple of 32 for resized width
# args['height'] = 1184 # nearest multiple of 32 for resized height
# args["east"] = "/content/textrecog/frozen_east_text_detection.pb" # path to input EAST text detector
# args["padding"] = 0.0 # amount of padding to add to each border of ROI
args["min_confidence"] = 0.0005 # minimum probability required to inspect a region

number_pattern = re.compile("^[0-4]{1}[0-9]{3}$") # Pattern to ensure value is likely a bib-number

image_file_titles = os.listdir(image_path)
image_file_titles.sort()

images = [os.path.join(image_path, img) for img in image_file_titles][500:550]

# Main

## Notebook Execution

In [ ]:
# Define arguments
args = {}
image_path = "./images"

# args['width'] = 800 # nearest multiple of 32 for resized width
# args['height'] = 1184 # nearest multiple of 32 for resized height
# args["east"] = "/content/textrecog/frozen_east_text_detection.pb" # path to input EAST text detector
# args["padding"] = 0.0 # amount of padding to add to each border of ROI
args["min_confidence"] = 0.0005 # minimum probability required to inspect a region

number_pattern = re.compile("^[0-4]{1}[0-9]{3}$") # Pattern to ensure value is likely a bib-number

image_file_titles = os.listdir(image_path)
image_file_titles.sort()

images = [os.path.join(image_path, img) for img in image_file_titles]

In [ ]:
reader = easyocr.Reader(['en'], gpu=True)

### Bib OCR: Doing it MY way

In [ ]:
images_to_numbers = {img: set() for img in images}
images_to_faces = {img: set() for img in images}

In [ ]:
images_to_numbers

{'./images\\20231123-BI6I0015.jpg': set(),
 './images\\20231123-BI6I0017.jpg': set(),
 './images\\20231123-BI6I0017_low_rez_.jpg': set(),
 './images\\20231123-BI6I0018.jpg': set(),
 './images\\20231123-BI6I0018_low_rez_.jpg': set(),
 './images\\20231123-BI6I0022.jpg': set(),
 './images\\20231123-BI6I0023.jpg': set(),
 './images\\20231123-BI6I0025.jpg': set(),
 './images\\20231123-BI6I0026.jpg': set(),
 './images\\20231123-BI6I0027.jpg': set(),
 './images\\20231123-BI6I0028.jpg': set(),
 './images\\f0341261-800px-wm.jpg': set()}

In [ ]:
for image in tqdm(images):
  results = reader.readtext(image)
  for (coords, text, confidence) in results:
    if confidence >= args['min_confidence']:
      if re.search(number_pattern, text): # Check if text is a number
        images_to_numbers[image].add(text)
        # print(text) # remove this later please

100%|██████████| 12/12 [00:07<00:00,  1.56it/s]


In [ ]:
to_lists = {pic: list(nums) for (pic,nums) in images_to_numbers.items()} # Convert sets back to lists for each element
json_output = json.dumps(to_lists, indent=1) # Make them easier to read, maybe

In [ ]:
with open(os.path.join("./images_to_numbers_easyocr.json"), "w") as f:
  f.write(json_output)

In [ ]:
images_to_numbers = None
with open(os.path.join("./images_to_numbers_easyocr.json")) as f:
  images_to_numbers = json.load(f)

image_file_titles = os.listdir(image_path)

In [ ]:
for image, numbers in tqdm(images_to_numbers.items()):
    file = os.path.join(".\\images", image.split('\\')[-1])
    print(file)
    info = IPTCInfo(file)
    info['keywords'] = [num for num in numbers]
    info.save(options=["overwrite"])

  0%|          | 0/12 [00:00<?, ?it/s]File not a JPEG, trying blindScan
No IPTC data found in .\images\20231123-BI6I0015.jpg
  0%|          | 0/12 [00:00<?, ?it/s]

.\images\20231123-BI6I0015.jpg


AttributeError: 'IPTCInfo' object has no attribute '_fob'

Testing accuracy

In [ ]:
image = image_path + '/20231123-BI6I0002.jpg'
IPTCInfo(image)['keywords']

FileNotFoundError: ignored

In [ ]:
!find "/content/drive/MyDrive/TurkeyTrot - 2023/Alan Pictures/images" ! -name '*~' | zip -r "/content/drive/MyDrive/TurkeyTrot - 2023/Alan Pictures/images.zip" -@



# !zip -r "/content/drive/MyDrive/TurkeyTrot - 2023/Alan Pictures/images.zip" "/content/drive/MyDrive/TurkeyTrot - 2023/Alan Pictures/images"

